# Chapter 3

Add your content here.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# --- Setup ---
# A problem requires 'DEPTH' correct steps to be solved.
DEPTH = 3 
# The probability a specific model gets a single step correct
STEP_ACCURACY = 0.6 

def simulate_step(accuracy):
    """Simulates one reasoning step. Returns True if correct."""
    return np.random.random() < accuracy

# --- Strategy 1: Greedy / One-Shot ---
def run_greedy(accuracy, depth):
    # To succeed, ALL steps must be correct sequentially
    for _ in range(depth):
        if not simulate_step(accuracy):
            return False # Failed at this step
    return True # Survived all steps

# --- Strategy 2: Best-of-N ---
def run_best_of_n(n, accuracy, depth):
    # We run 'n' independent full attempts.
    # If ANY of them succeeds, the problem is solved.
    # (Assumes we have a perfect verifier to pick the right one)
    for _ in range(n):
        if run_greedy(accuracy, depth):
            return True
    return False

# --- Strategy 3: Tree Search (Simplified) ---
def run_tree_search(budget_nodes, accuracy, depth):
    """
    A simplified Breadth-First Search.
    We verify each step before moving to the next.
    'budget_nodes' is the max number of generation calls we can make.
    """
    # Current active distinct valid paths at the current depth
    current_paths = 1 
    
    nodes_used = 0
    
    for d in range(depth):
        # For the current level, we try to expand our valid paths.
        # If we fail a step, we can retry if we have budget.
        
        next_level_paths = 0
        
        # We try to advance each valid path
        # In a real tree, we might branch. Here, let's assume we just need
        # to find ONE valid node at the next layer to proceed.
        
        while nodes_used < budget_nodes:
            nodes_used += 1
            if simulate_step(accuracy):
                next_level_paths = 1
                break # Found a valid step for this level! Move to next level.
        
        if next_level_paths == 0:
            return False # Ran out of budget before solving this level
            
    return True

# --- Experiment ---
# Compare Best-of-N vs Tree Search at iso-compute budgets
budgets = [1, 5, 10, 20, 50, 100] # Number of total model calls allowed
bon_results = []
tree_results = []

NUM_TRIALS = 1000

print(f"{'Budget':<10} | {'Best-of-N Acc':<15} | {'Tree Search Acc':<15}")
print("-" * 50)

for b in budgets:
    # Test BoN
    bon_wins = sum([run_best_of_n(b, STEP_ACCURACY, DEPTH) for _ in range(NUM_TRIALS)])
    bon_acc = bon_wins / NUM_TRIALS
    
    # Test Tree Search
    # Note: For Tree Search, budget is total nodes.
    tree_wins = sum([run_tree_search(b, STEP_ACCURACY, DEPTH) for _ in range(NUM_TRIALS)])
    tree_acc = tree_wins / NUM_TRIALS
    
    bon_results.append(bon_acc)
    tree_results.append(tree_acc)
    
    print(f"{b:<10} | {bon_acc:<15.3f} | {tree_acc:<15.3f}")

# Plotting (In a real scenario)
# You would see Tree Search rising to 1.0 (100%) much faster than Best-of-N.

Budget     | Best-of-N Acc   | Tree Search Acc
--------------------------------------------------
1          | 0.219           | 0.000          
5          | 0.705           | 0.701          
10         | 0.916           | 0.991          
20         | 0.987           | 1.000          
50         | 1.000           | 1.000          
100        | 1.000           | 1.000          
